In [ ]:
from pathlib import Path

from meu_danfe import MeuDanfeClient

# Was: a hand-rolled sync `requests` client (LiveServerSession,
# find_and_add_nfe, get_nfe with a bare `except:` that swallowed every
# error including KeyboardInterrupt) — now just the library.
with MeuDanfeClient.from_env() as client:
    document = client.fetch("CHAVE_DE_ACESSO_AQUI")
    print(document.name, document.doc_type, len(document.data))

In [ ]:
"""
Série -> data['nfeProc']['NFe']['infNFe']['ide']['serie']
Data de Emissão -> data['nfeProc']['NFe']['infNFe']['ide']['dhEmi']
Data de Saída/Entrada -> data['nfeProc']['NFe']['infNFe']['ide']['dhSaiEnt']
CNPJ Emitente -> data['nfeProc']['NFe']['infNFe']['emit']['CNPJ']
Razão Social Emitente -> data['nfeProc']['NFe']['infNFe']['emit']['xNome']
UF Emitente -> data['nfeProc']['NFe']['infNFe']['emit']['enderEmit']['UF']
CNPJ Destinatário -> data['nfeProc']['NFe']['infNFe']['dest']['CNPJ']
Razão Social Destinatário -> data['nfeProc']['NFe']['infNFe']['dest']['xNome']
Natureza da Operação -> data['nfeProc']['NFe']['infNFe']['ide']['natOp']
Sequencial do Item -> data['nfeProc']['NFe']['infNFe']['det'][0]['nItem']
Código do Produto -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['cProd']
Descrição do Produto -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['xProd']
NCM -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['NCM']
CFOP -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['CFOP']
Unidade Comercial -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['uCom']
Quantidade -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['qCom']
Valor Unitário -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['vUnCom']
Valor Total Bruto do Item -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['vProd']
CST/CSOSN -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['imposto']['ICMS']
Base de Cálculo ICMS -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']['ICMS']
Alíquota ICMS -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']['ICMS']
Valor ICMS -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']['ICMS']
Base de Cálculo ICMS ST -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']
Valor ICMS ST -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']
Valor IPI -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']
Base de Cálculo PIS/COFINS -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']
Alíquotas PIS/COFINS -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']
Valores PIS/COFINS -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']
Valor Total da Nota -> data['nfeProc']['NFe']['infNFe']['total']['ICMSTot']['vNF']
Valor Total dos Produtos -> data['nfeProc']['NFe']['infNFe']['total']['ICMSTot']['vProd']
Frete -> data['nfeProc']['NFe']['infNFe']['total']['ICMSTot']['vFrete']
Seguro -> data['nfeProc']['NFe']['infNFe']['total']['ICMSTot']['vSeg']
Desconto -> data['nfeProc']['NFe']['infNFe']['total']['ICMSTot']['vDesc']
Informações Complementares -> data['nfeProc']['NFe']['infNFe']['infAdic']['infCpl']
"""
# Fixed: CNPJ Destinatário / Razão Social Destinatário were swapped in this
# doc cell (src/meu_danfe/nfe/columns.json already had them right).

# Filter

In [ ]:
import pandas as pd
import shutil
import re

todo = pd.read_csv('../data/notas.tsv', sep='\t')
todo.PROJETO.unique()

WD = Path('/home/leonardo/Downloads/OneDrive_2026-04-25/02. Nacional/')
files = []
for item in WD.glob('*'):
    if item.is_dir() and item.name in todo.PROJETO.values:
        group = todo[todo.PROJETO == item.name]
        r = re.compile('|'.join(group.NOTAS.astype(str)))
        for file in item.rglob('**/*'):
            if re.search(r, file.name):
                files.append({'PROJECT': item.name, 'FILE': file})
todo.dtypes
found = pd.DataFrame(files)
found['NUMBER'] = found['FILE'].map(lambda x: re.search(r'\d{3,}', x.name).group())
found.query('PROJECT == "ANAGÉ"')
found['NUMBER'] = found['NUMBER'].astype(int)
todo.merge(found, how='left_anti', left_on=['PROJETO', 'NOTAS'], right_on=['PROJECT', 'NUMBER'])

# Key extraction from DANFE PDFs now lives in meu_danfe.pdf
# (extract_keys / extract_keys_from_dir) — the duplicate extract_key()
# that used to be defined here (and was never even called) has been removed.
# from meu_danfe.pdf import extract_keys_from_dir
# by_file = extract_keys_from_dir(WD)

todo['CNPJ'] = todo['CNPJ'].str.replace(r'[^0-9]', '', regex=True)
chaves = pd.read_csv('../data/chaves_nfe.csv', header=None, names=['chave'], dtype=str)
chaves[['cnpj', 'numero']] = chaves['chave'].str.extract(r'^\d{6}(\d{14})\d{5}(\d{9})', expand=True)
chaves['numero'] = chaves['numero'].astype(int)
todo.merge(
    chaves, how="inner", left_on=["CNPJ", "NOTAS"], right_on=["cnpj", "numero"]
).drop_duplicates()['chave'].to_csv('../data/todo.csv', index=False)